# Replicant — Protocol Metrics (OTel JSON)

Reads per-scenario OTel JSON files written by the orchestrator's `--metrics-file` flag and produces protocol-level figures: sync traffic, op latency, post-convergence doc size.

**`in_process` only.** The `--metrics-file` flag captures meaningful data only when the replicas run inside the orchestrator process — so their instruments register with the orchestrator's `MeterProvider`. For `docker` / `k8s` runs the replicas are separate processes and export via OTLP → collector → Prometheus instead; that path is covered by [`live_metrics.ipynb`](live_metrics.ipynb).

**Generate inputs first** (release build, one scenario at a time — OTel counters accumulate across all scenarios in a single orchestrator process, so a multi-scenario run produces only cumulative values):

```sh
mkdir -p results
for s in $(ls scenarios/*.toml | xargs -n1 basename -s .toml); do
  cargo run --release --bin orchestrator -- --trials 10 \
    --metrics-file "results/metrics-${s}.json" \
    --output csv "scenarios/${s}.toml" \
    > /dev/null 2>&1
done
```

The cross-scenario `sync_df` and `doc_wide` cells iterate the scenarios listed in `results/results.csv`, so the in_process orchestrator run that populated that CSV must also have produced the per-scenario metrics files.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150

REPO    = Path("..").resolve()
RESULTS = REPO / "results"
CSV     = RESULTS / "results.csv"  # in_process only — see header
FIGS    = REPO / "analysis" / "figures" / "in_process"
FIGS.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the in_process summary table — the cross-scenario cells below
# iterate `summary.scenario.unique()` to find each metrics-<scenario>.json.
df = pd.read_csv(CSV)
summary = df[df.row_type == "summary"].rename(columns={"trial": "n_trials"})
print(f"{summary.scenario.nunique()} scenarios in {CSV.name}")

## OTel Protocol Metrics

> **`SOURCE = "in_process"` only.** This section reads JSON files emitted by the orchestrator's `--metrics-file` flag, which only captures meaningful data when the replicas run *inside* the orchestrator process (so the replica-side instruments register with the orchestrator's `MeterProvider`). For `SOURCE = "docker"` or `"k8s"`, the replicas live in separate processes and export OTLP to a collector → Prometheus instead, covered by the "Prometheus-backed metrics (live stack)" section below. The cells in this section are no-ops when the `metrics-*.json` files don't exist.

Loaded from per-scenario `results/metrics-<scenario>.json` files written by the orchestrator when `--metrics-file` is passed.

OTel counters accumulate across every scenario in a single run, so **run one scenario at a time** to get per-scenario metrics. The loop below covers every TOML in `scenarios/`:

```sh
mkdir -p results
for s in $(ls scenarios/*.toml | xargs -n1 basename -s .toml); do
  cargo run --release --bin orchestrator -- --trials 10 \
    --metrics-file "results/metrics-${s}.json" \
    "scenarios/${s}.toml" \
    > /dev/null 2>&1
done
```

The `Sync traffic per write op` cell below reads every `results/metrics-<scenario>.json` and produces a cross-topology comparison.

Point `METRICS` below at any single file (e.g. `RESULTS / "metrics-full-mesh-n5.json"`) to inspect one scenario's per-node breakdown.

> **Multi-scenario (cumulative) alternative** — useful only as a sanity check that overall protocol traffic looks reasonable; values cannot be attributed to individual scenarios:
> ```sh
> cargo run --release --bin orchestrator -- --trials 10 --output csv \
>   --metrics-file results/metrics-all.json \
>   scenarios/*.toml \
>   2>/dev/null > results/results.csv
> ```

In [ ]:
import json

# Point this at a single-scenario metrics file for per-scenario analysis,
# e.g. RESULTS / "metrics-full-mesh-n5.json".
# Use RESULTS / "metrics-all.json" for the cumulative multi-scenario view.
METRICS = RESULTS / "metrics.json"


def load_metrics(path: Path) -> dict[str, pd.DataFrame] | None:
    """Parse a metrics JSON Lines file into a dict of DataFrames keyed by metric name.

    Each line in the file is a JSON object ``{"metrics": [...]}``.
    Data points from multiple flushes are concatenated per metric name.
    Returns None if the file does not exist.
    """
    if not path.exists():
        print(f"[metrics] {path} not found — run the per-scenario loop above first.")
        return None

    rows: list[dict] = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    by_name: dict[str, list[dict]] = {}
    for record in rows:
        for m in record.get("metrics", []):
            by_name.setdefault(m["name"], []).extend(m["data_points"])

    return {name: pd.DataFrame(points) for name, points in by_name.items()}


metrics = load_metrics(METRICS)
if metrics:
    print("Loaded metrics:", list(metrics.keys()))
    for name, df_m in metrics.items():
        print(f"  {name}: {len(df_m)} data points, columns={list(df_m.columns)}")

### Sync message traffic

Total Automerge sync messages sent and received per node.
Shows how much protocol chatter each node generates — useful for arguing O(N²) scaling of the gossip layer.


In [ ]:
if metrics:
    tx = metrics["replicant.sync.messages.tx"].groupby("actor")["value"].sum().rename("tx")
    rx = metrics["replicant.sync.messages.rx"].groupby("actor")["value"].sum().rename("rx")
    traffic = pd.concat([tx, rx], axis=1).fillna(0).astype(int)
    # Sort nodes naturally (node-0, node-1, …)
    traffic = traffic.loc[sorted(traffic.index, key=lambda s: int(s.split("-")[1]))]

    fig, ax = plt.subplots(figsize=(max(4, len(traffic) * 0.9), 4))
    x = range(len(traffic))
    width = 0.35
    ax.bar([i - width / 2 for i in x], traffic["tx"], width, label="sent (tx)", alpha=0.8)
    ax.bar([i + width / 2 for i in x], traffic["rx"], width, label="received (rx)", alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(traffic.index)
    ax.set_xlabel("Node")
    ax.set_ylabel("Sync messages")
    ax.set_title("Sync message traffic per node (all scenarios, cumulative)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGS / "sync_traffic.pdf")
    plt.show()
    print(traffic.assign(total=traffic.tx + traffic.rx).sort_values("total", ascending=False))


## Sync traffic per write op — by scenario

Loads every per-scenario `metrics-<scenario>.json` file (one per scenario, generated by the loop in the markdown above) and sums `tx + rx` across all nodes. Dividing by `trials × op_count` yields **sync messages per write op**, comparable across scenarios with different N and op_count.

This quantifies the **amplification cost** of Phase A's relay-on-receive fix: more edges (higher topology connectivity) → more redundant fan-out per write. It explains why full-mesh-n10 (45 edges) converges slower than line-n10 (9 edges) despite the shorter diameter — the line has 10× fewer message paths to flood.

In [ ]:
import matplotlib.patches as mpatches

sync_rows = []
for scen in summary.scenario.unique():
    mpath = RESULTS / f"metrics-{scen}.json"
    m = load_metrics(mpath)
    if m is None or "replicant.sync.messages.tx" not in m:
        continue
    tx_total = int(m["replicant.sync.messages.tx"]["value"].sum())
    rx_total = int(m["replicant.sync.messages.rx"]["value"].sum())
    s_row = summary[summary.scenario == scen].iloc[0]
    sync_rows.append({
        "scenario": scen,
        "topology_kind": s_row.topology_kind,
        "node_count": int(s_row.node_count),
        "edge_count": int(s_row.edge_count),
        "diameter": int(s_row.diameter),
        "msgs_per_op": (tx_total + rx_total) / (s_row.n_trials * s_row.op_count),
    })
sync_df = pd.DataFrame(sync_rows).sort_values(["topology_kind", "node_count"]).reset_index(drop=True)

kinds = sorted(sync_df.topology_kind.unique())
palette = dict(zip(kinds, sns.color_palette("muted", n_colors=len(kinds))))
bar_colors = [palette[k] for k in sync_df.topology_kind]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(sync_df.scenario, sync_df.msgs_per_op,
       color=bar_colors, edgecolor="black", linewidth=0.4, alpha=0.9)
ax.tick_params(axis="x", rotation=45)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
ax.set_xlabel(None)
ax.set_ylabel("Sync messages per write op (tx + rx, all nodes)")
ax.set_title("Relay amplification cost by scenario")
legend_handles = [mpatches.Patch(color=palette[k], label=k) for k in kinds]
ax.legend(handles=legend_handles, title="Topology", loc="upper left")
fig.tight_layout()
fig.savefig(FIGS / "sync_vs_topology.pdf")
plt.show()
sync_df.round(2)

### Op application latency

Per-node mean latency for `map_put` operations (from the histogram `sum / count`).
Measures the cost of Automerge's local commit — independent of network.


In [ ]:
if metrics:
    op_df = metrics["replicant.op.duration"].copy()
    op_df["mean_ms"] = op_df["sum"] / op_df["count"]
    op_df = op_df.sort_values("actor", key=lambda s: s.map(lambda v: int(v.split("-")[1])))

    fig, ax = plt.subplots(figsize=(max(4, len(op_df) * 0.9), 4))
    bars = ax.bar(op_df["actor"], op_df["mean_ms"], alpha=0.8)
    if "min" in op_df.columns and "max" in op_df.columns:
        ax.errorbar(
            op_df["actor"],
            op_df["mean_ms"],
            yerr=[op_df["mean_ms"] - op_df["min"], op_df["max"] - op_df["mean_ms"]],
            fmt="none",
            color="black",
            capsize=4,
        )
    ax.set_xlabel("Node")
    ax.set_ylabel("Mean op latency (ms)")
    ax.set_title("Automerge map_put latency per node (min/mean/max)")
    fig.tight_layout()
    fig.savefig(FIGS / "op_latency.pdf")
    plt.show()
    print(op_df[["actor", "count", "min", "mean_ms", "max"]].to_string(index=False))


### Document size after convergence — per-scenario per-node

For every scenario, the table below lists the serialized Automerge document size on each replica at scenario-end. The orchestrator's convergence check already enforces the **CRDT invariant** — every replica must reach the same logical state, i.e. byte-equal `state_fingerprint()` (sorted change hashes). Without that, no metrics file would exist.

What this table shows is something a strict CRDT check *doesn't* catch: Automerge's `save()` byte output is **not canonical** across replicas with identical logical state. Two replicas with the same DAG and the same readable values can produce different byte counts because the encoding preserves change-list storage order, and that order depends on the order each replica integrated remote changes.

Empirically (locked in by `adapter::tests::save_bytes_not_canonical_across_converged_replicas`):

- **Concentrated writes (any topology, including partition-heal)**: spread = 0. With a single author per replica's local history — node-0 for the full-mesh/line/ring/star cases, `group.nodes[0]` of each group for partition-heal — replicas integrate every change in the same author order, and `save()` produces byte-identical output. This is structurally guaranteed, not stochastic.
- **Round-robin on multi-author topologies**: spread > 0, but small (a few percent). The size depends on stochastic receive timing: replicas at different graph positions see remote changes in different orders. The full-mesh case is *not* always 0 — at n=10 with 20 writes scattered across 10 authors there's enough interleaving for timing-induced reorders to surface in the encoding (~4-5% spread). Smaller-N full-mesh runs often round to spread=0 simply because the change count is small and the receive order happens to align.

The `spread_pct` column is the right way to read this: a few percent is the encoding-ordering effect; anything larger would indicate a real timing or sync bug worth investigating (asserted below at the 10% threshold).

Empty cells (`—`) mean that scenario uses fewer nodes than the maximum in the suite.

In [ ]:
from IPython.display import display

doc_rows = []
missing = []
for scen in summary.scenario.unique():
    mpath = RESULTS / f"metrics-{scen}.json"
    if not mpath.exists():
        missing.append(scen)
        continue
    m = load_metrics(mpath)
    if m is None or "replicant.doc.size_bytes" not in m:
        missing.append(scen)
        continue
    # The gauge is re-sampled on every local op and every sync_receive, so
    # each actor appears in many rows. Keep the last flush per actor — by
    # scenario-end the cluster is converged (fingerprints match), so that
    # sample is each replica's post-convergence save() length.
    latest = m["replicant.doc.size_bytes"].drop_duplicates("actor", keep="last")
    for _, r in latest.iterrows():
        doc_rows.append({"scenario": scen, "actor": r["actor"], "bytes": int(r["value"])})

if not doc_rows:
    print("[doc_size] no per-scenario metrics files found — re-run scenarios with --metrics-file.")
else:
    doc_long = pd.DataFrame(doc_rows)
    node_cols = sorted(doc_long.actor.unique(), key=lambda s: int(s.split("-")[1]))
    doc_wide = doc_long.pivot(index="scenario", columns="actor", values="bytes").reindex(columns=node_cols)
    doc_wide["min"] = doc_wide.min(axis=1).astype(int)
    doc_wide["max"] = doc_wide.max(axis=1).astype(int)
    doc_wide["spread"] = (doc_wide["max"] - doc_wide["min"]).astype(int)
    doc_wide["spread_pct"] = (doc_wide["spread"] / doc_wide["min"] * 100).round(1)

    # Soft check: a few percent is the Automerge save()-ordering effect
    # documented in the markdown above; anything much larger would indicate
    # a real sync/timing bug. The CRDT convergence invariant itself is the
    # fingerprint match enforced by the orchestrator — see the
    # `adapter::tests::save_bytes_not_canonical_across_converged_replicas`
    # test that locks this property in.
    SPREAD_PCT_THRESHOLD = 10.0
    suspicious = doc_wide[doc_wide["spread_pct"] > SPREAD_PCT_THRESHOLD]
    assert suspicious.empty, (
        f"doc_size spread exceeds {SPREAD_PCT_THRESHOLD}% in: "
        f"{suspicious[['min', 'max', 'spread', 'spread_pct']].to_dict('index')}"
    )

    if missing:
        print(f"[doc_size] skipped {len(missing)} scenarios with no metrics file: {missing}")
    n_canonical = int((doc_wide["spread"] == 0).sum())
    print(f"{n_canonical}/{len(doc_wide)} scenarios have byte-identical save() across replicas")
    print(f"max spread observed: {int(doc_wide['spread'].max())} bytes "
          f"({doc_wide['spread_pct'].max():.1f}%)\n")

    # display() instead of last-expression rendering — the table is nested
    # inside `else:` so Jupyter's auto-render doesn't fire. Sort by spread
    # descending so the non-canonical scenarios surface at the top.
    display(doc_wide.sort_values("spread", ascending=False).fillna("—"))